# Khảo Sát và Khám Phá Dữ Liệu Bất Động Sản (Exploratory Data Analysis - EDA)

**Đề tài:** Xây dựng bộ dữ liệu về bất động sản, thực hiện các phương pháp tối ưu hiệu năng cho hệ thống khi dữ liệu quá lớn.
**Quy mô dữ liệu:** 500,000 bản ghi thực tế từ Hugging Face `Phuscca/vietnam-real-estates`.
**Mục tiêu nghiên cứu:** Phân tích cấu trúc phân phối, kiểm chứng dữ liệu khuyết thiếu, phát hiện điểm dị biệt và xác lập nền tảng đặc trưng (Feature Engineering) phục vụ tối ưu hóa truy vấn CSDL PostgreSQL.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Thiết lập giao diện đồ họa chuẩn học thuật
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Segoe UI"]
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_PATH = "../data/processed/properties_500000.csv"
print(f"Dang doc du lieu tu {DATA_PATH}...")
df = pd.read_csv(DATA_PATH)
print(f"So dong: {len(df):,}, So cot: {len(df.columns)}")
df.head()

## 1. Phân Tích Dữ Liệu Khuyết Thiếu (Missing Values Analysis)

Đánh giá mức độ hoàn thiện của từng cột dữ liệu. Chú ý thuộc tính `price` có giá trị thiếu phản ánh các tin đăng có giá bán ở trạng thái "Thỏa thuận".

In [ ]:
missing = pd.DataFrame({
    "Total": len(df),
    "Missing_Count": df.isna().sum(),
    "Missing_Percent": (df.isna().mean() * 100).round(2)
})
missing.sort_values(by="Missing_Percent", ascending=False)

## 2. Thống Kê Mô Tả và Phân Bố Thuộc Tính Số Học (Price & Area)

Khảo sát các đại lượng thống kê tứ phân vị (Q1, Median, Q3), trung bình, độ lệch chuẩn và phân vị ngoại vi (P95, P99).

In [ ]:
num_cols = ["price", "area"]
df[num_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Phân bố giá dưới P99
valid_p = df["price"].dropna()
p99 = valid_p.quantile(0.99)
sns.histplot(valid_p[valid_p <= p99] / 1e9, bins=50, kde=True, ax=axes[0], color="#2b5c8f")
axes[0].set_title("Phân Bố Giá Bán (< 99th Percentile: 130 Tỷ VNĐ)")
axes[0].set_xlabel("Giá Bán (Tỷ VNĐ)")
axes[0].set_ylabel("Số Lượng")

# Phân bố Log10
sns.histplot(np.log10(valid_p[valid_p > 0]), bins=50, kde=True, ax=axes[1], color="#d95f02")
axes[1].set_title("Phân Bố Giá Bán Dạng Log10")
axes[1].set_xlabel("Log10(Giá VNĐ)")

plt.tight_layout()
plt.show()

## 3. Phân Tích Phân Loại Bất Động Sản và Địa Bàn Hành Chính

Khảo sát thị phần tin đăng theo các loại hình (Nhà riêng, Căn hộ, Đất nền, Biệt thự) và mật độ tin đăng theo tỉnh/thành phố.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Loại BĐS
ptypes = df["property_type_name"].value_counts().reset_index()
ptypes.columns = ["Type", "Count"]
sns.barplot(data=ptypes, x="Count", y="Type", palette="viridis", ax=axes[0])
axes[0].set_title("Cơ Cấu Loại Bất Động Sản")

# Top 10 Tỉnh/Thành
top_prov = df["province_name"].value_counts().head(10).reset_index()
top_prov.columns = ["Province", "Count"]
sns.barplot(data=top_prov, x="Count", y="Province", palette="mako", ax=axes[1])
axes[1].set_title("Top 10 Địa Phương Có Lượng Tin Đăng Lớn Nhất")

plt.tight_layout()
plt.show()

## 4. Tương Quan Giá và Diện Tích & Biến Động Theo Loại Hình

Sử dụng biểu đồ mật độ Hexbin để trực quan hóa dữ liệu lớn mà không bị chồng lấp (overplotting).

In [ ]:
# Lọc mẫu đại diện 20,000 điểm dữ liệu hợp lệ dưới P99 để trực quan
valid_subset = df.dropna(subset=["price", "area"]).copy()
p99_p = valid_subset["price"].quantile(0.99)
p99_a = valid_subset["area"].quantile(0.99)
filtered = valid_subset[(valid_subset["price"] <= p99_p) & (valid_subset["area"] <= p99_a)]
sample = filtered.sample(min(20000, len(filtered)), random_state=42)

fig, ax = plt.subplots(figsize=(10, 6))
hb = ax.hexbin(sample["area"], sample["price"] / 1e9, gridsize=35, cmap="Blues", bins="log")
cb = fig.colorbar(hb, ax=ax)
cb.set_label("Log10(Mật độ)")
ax.set_title("Quan Hệ Diện Tích (m²) và Giá Bán (Tỷ VNĐ)")
ax.set_xlabel("Diện Tích (m²)")
ax.set_ylabel("Giá Bán (Tỷ VNĐ)")
plt.show()

## 5. Phân Tích Chuỗi Thời Gian và Ma Trận Tương Quan Thứ Hạng

Kiểm tra tính mùa vụ theo tháng và tương quan Spearman giữa các biến số.

In [ ]:
corr_df = pd.DataFrame({
    "price": df["price"],
    "area": df["area"],
    "price_per_m2": df["price"] / df["area"]
})
dates = pd.to_datetime(df["published_at"], errors="coerce")
corr_df["year"] = dates.dt.year
corr_df["month"] = dates.dt.month

spearman_corr = corr_df.corr(method="spearman")
plt.figure(figsize=(7, 5))
sns.heatmap(spearman_corr, annot=True, cmap="coolwarm", fmt=".2f", square=True)
plt.title("Ma Trận Tương Quan Thứ Hạng Spearman")
plt.show()

## 6. Kết Luận Nghiệm Thu EDA

1. **Phân phối lệch phải cực mạnh:** Cả `price` và `area` đều có phân phối Pareto-like, với P50 = 7.4 tỷ VNĐ nhưng Mean = 20.78 tỷ VNĐ do sự xuất hiện của các bất động sản thương mại/dự án lớn.
2. **Tương quan thuận:** Tương quan Spearman giữa `area` và `price` đạt 0.58, khẳng định diện tích là nhân tố quyết định bậc nhất tới giá thành.
3. **Tập trung hóa vùng địa lý:** Hà Nội và TP.HCM chiếm đa số tuyệt đối tổng lượng tin đăng, là cơ sở khoa học cho bài toán lập chỉ mục theo `province_name` (Q1) và truy vấn kết hợp lọc địa lý - sắp xếp thời gian (Q2).
4. **Cơ sở cho Feature Engineering:** Sự biến thiên mạnh mẽ của đơn giá trên m² giữa các tỉnh cho thấy `price_per_m2` là chỉ số chuẩn hóa hữu hiệu nhất để phân tích định giá thị trường.